In [ ]:
import scanpy as sc
import scvelo as scv
import pandas as pd
import numpy as np

In [ ]:
scv.__version__

In [ ]:
adata_raw = sc.read_h5ad('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P2/data/scMulti-omics/1.hscAdata_raw.h5ad')
# previous label error 
adata_raw.obs['sample2'] = adata_raw.obs['Sample'].replace({
    'GW7': 'GW6',
    'GW6': 'GW7'
})
adata_raw.obs['barcode2'] = adata_raw.obs_names.str.rsplit('-', n=1).str[0]
adata_raw

In [ ]:
mat = pd.read_csv('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/Annotation/metadata_snRNA.csv', header=0, index_col=0)
mat

In [ ]:
# 创建匹配键：将barcode2和sample2组合成唯一的标识符
# 需要将category类型转换为字符串类型
adata_raw.obs['match_key'] = adata_raw.obs['barcode2'].astype(str) + '_' + adata_raw.obs['sample2'].astype(str)
mat['match_key'] = mat['barcode2'].astype(str) + '_' + mat['sample2'].astype(str)

# 获取在mat中存在的match_key
existing_match_keys = set(mat['match_key'])

# 筛选出在mat中存在的细胞
mask = adata_raw.obs['match_key'].isin(existing_match_keys)
adata_filtered = adata_raw[mask, :].copy()

# 一次性从mat中提取cell_type和UMAP坐标
# 创建包含所有需要信息的映射字典
info_mapping = mat.set_index('match_key')[['cell_type_snRNA', 'UMAP_1', 'UMAP_2']].to_dict('index')

# 提取cell_type和UMAP坐标
cell_names = []
umap_coords = []

for match_key in adata_filtered.obs['match_key']:
    if match_key in info_mapping:
        info = info_mapping[match_key]
        cell_names.append(info['cell_type_snRNA'])
        umap_coords.append([info['UMAP_1'], info['UMAP_2']])
    else:
        # 理论上不会进入这里，因为已经筛选过了，但为了安全还是处理
        cell_names.append(None)
        umap_coords.append([np.nan, np.nan])

# 同时添加到obs和obsm中
adata_filtered.obs['cell_name'] = cell_names
adata_filtered.obsm['X_umap'] = np.array(umap_coords)

# 删除临时的match_key列
adata_filtered.obs.drop('match_key', axis=1, inplace=True)

In [ ]:
# 定义映射字典
map_dict = {
    "AST": "AST_dorsal_2",
    "COP": "COP",
    "ExN_immature_3": "ExN_dorsal_3",
    "IPC": "IPC",
    "APC_dorsal": "AST_dorsal_1",
    "APC_ventral": "AST_ventral",
    "ExN_mature_2": "ExN_dorsal_5",
    "ExN_immature_2": "ExN_dorsal_2",
    "ExN_mature_1": "ExN_dorsal_4",
    "ExN_immature_1": "ExN_dorsal_1",
    "ExN_naive": "ExN_naive",
    "ExN_mature_3": "ExN_dorsal_6",
    "FP": "FP",
    "InN_immature_1": "InN_dorsal_1",
    "InN_immature_2": "InN_dorsal_2",
    "InN_mature_1": "InN_dorsal_3",
    "InN_mature_2": "InN_dorsal_4",
    "InN_naive": "InN_naive",
    "InN_ventral_1": "InN_ventral_1",
    "MFOL": "MFOL",
    "Microglia": "Microglia",
    "MN1": "MN1",
    "MN2": "MN2",
    "MOL": "MOL",
    "NPC": "NPC",
    "NPC_proliferative": "NPC_proliferative",
    "OPC": "OPC",
    "RP": "RP",
    "MP": "MP",
    "FP": "FP",
    "InN_ventral_2": "InN_ventral_2",
    "ExN_ventral": "ExN_ventral",
    "ventral_glia": "AST_ventral",
    "InN_ventral_3": "InN_ventral_3",
    "APC_cycling": "APC",
    "VLMC": "VLMC",
    "Schwann": "Schwann",
    "EC": "EC",
    "PC": "PC"

}

# 使用map函数创建新列'celltype'
adata_filtered.obs['celltype'] = adata_filtered.obs['cell_name'].map(map_dict)

In [ ]:
sc.pl.umap(
    adata_filtered,
    color=["celltype"],
    legend_loc="on data",
)

In [ ]:
sc.pl.umap(
    adata_filtered,
    color=["PIEZO1"],
    legend_loc="on data",
    vmax=2
)

In [ ]:
target_celltypes = ["NPC", "IPC", "ExN_naive", "InN_naive", "OPC", "APC"]

# 筛选adata对象中指定细胞类型的细胞
adata = adata_filtered[adata_filtered.obs['celltype'].isin(target_celltypes)].copy()
adata.layers["counts"] = adata.X.copy()
sc.pp.scrublet(adata, batch_key="sample2")
adata = adata[adata.obs['predicted_doublet'] == False, :].copy()
# adata = adata[adata.obs['doublet_score'] <= 0.2, :].copy()
adata

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000, batch_key="sample2")
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30)

In [ ]:
sc.external.pp.bbknn(adata, 
                     batch_key="sample2",
                     n_pcs = 30,
                     use_annoy= False,
                     pynndescent_n_neighbors = 30
                    ) 

sc.tl.umap(adata, 
           min_dist = 0.3, 
           spread = 1.5)

sc.pl.umap(
    adata,
    color=["celltype"],
    legend_loc="on data")

In [ ]:
sc.pl.umap(
    adata,
    color=["TFDP2", "PAX3", "NFIB", "ASCL1", 'EGFR', 'ZBTB20', 'GFAP'],
    legend_loc="on data",
    vmax=2
)

In [ ]:
sc.pl.umap(
    adata,
    color=["MEIS1"],
    legend_loc="on data",
    vmax=2
)

In [ ]:
sc.tl.leiden(adata, flavor="igraph", n_iterations=2, resolution=2)
sc.pl.umap(adata, color=["leiden"], legend_loc="on data")

In [ ]:
clusters_to_exclude = ['1', '15']
keep_cell = [cluster not in clusters_to_exclude for cluster in adata.obs['leiden']]
adata = adata[keep_cell, :].copy()

In [ ]:
adata

In [ ]:
# Normalizing to median total counts
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30)

In [ ]:
sc.external.pp.bbknn(adata, 
                     batch_key="sample2",
                     n_pcs = 30,
                     use_annoy= False,
                     pynndescent_n_neighbors = 30
                    )  # running bbknn 1.3.6

sc.tl.umap(adata, 
           min_dist = 0.3, 
           spread = 1.2)

sc.pl.umap(
    adata,
    color=["celltype"],
    legend_loc="on data",
)


In [ ]:
adata

In [ ]:
sc.pl.umap(
    adata,
    color=["EGFR", "ASCL1", "PAX3", "NFIB", "ZBTB20", "TFDP2", "GFAP"],
    legend_loc="on data",
    vmax=2
)

In [ ]:
ldata_combined = sc.read('/cluster2/huanglab/jiamao/Project/Tools/RNAvelocity/1.hscLoom_raw.h5ad')

In [ ]:
cellselect = adata.obs.index.tolist()
# cellselect = adata[adata.obs['sample2'] != 'GW17'].obs.index.tolist()
# cellselect = adata[adata.obs['sample2'] == 'GW17'].obs.index.tolist()
cellselect

In [ ]:
filtered_ldata = ldata_combined[ldata_combined.obs.index.isin(cellselect)]
filtered_ldata

In [ ]:
svdata = scv.utils.merge(adata, filtered_ldata)

In [ ]:
scv.pp.filter_and_normalize(svdata, min_shared_counts=10, n_top_genes=6000)
sc.pp.pca(adata)
# sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(svdata, n_pcs=None, n_neighbors=None)
# compute velocity 
scv.tl.velocity(svdata, mode='stochastic')
scv.tl.velocity_graph(svdata)
scv.tl.terminal_states(svdata)

In [ ]:
scv.set_figure_params(figsize=(6, 6))

In [ ]:
palette={
    'NPC': '#faa275', 
    'IPC': '#34a0a4',
    'OPC': '#168aad', 
    'ExN_naive': '#BF3131',
    'InN_naive': '#BA6EAC',
    'APC': '#D15500'
}

scv.pl.velocity_embedding_stream(svdata, 
                                 basis='umap',
                                 color="celltype", 
                                 smooth=0.5,
                                 min_mass=1, 
                                 cutoff_perc= 0, 
                                 palette = palette, 
                                #  legend_loc="right margin",
                                 size=20, 
                                 alpha=1, 
                                 arrow_size=0.5, 
                                 linewidth=0.4, 
                                 dpi=400, 
                                 save='velocity_embedding_stream2.svg'
                                 )

In [ ]:
svdata

In [ ]:
scv.pl.velocity_embedding_grid(svdata, 
                                 basis='umap',
                                 color="celltype", 
                                 smooth=0.5,
                                 min_mass=1, 
                                #  cutoff_perc= 0, 
                                 palette = palette, 
                                #  legend_loc="right margin", 
                                 size=20, 
                                 alpha=1, 
                                 arrow_size=1,
                                 linewidth=0.15, 
                                 dpi=400, 
                                 save='velocity_embedding_grid2.svg'
                                 )

In [ ]:
svdata

In [ ]:
# 查看自动计算的 root_cells
scv.pl.scatter(svdata, color='end_points', color_map='Reds')

In [ ]:
svdata

In [ ]:
scv.tl.recover_dynamics(svdata)
scv.tl.velocity(svdata, mode='dynamical')
scv.tl.velocity_graph(svdata)
scv.tl.latent_time(svdata)

In [ ]:
scv.pl.scatter(svdata, 
               color='latent_time', 
               color_map='gnuplot', 
               size=80, 
               colorbar=True)

In [ ]:
scv.pl.velocity_embedding_grid(svdata, 
                                 basis='umap',
                                 color="latent_time", 
                                 smooth=0.5,
                                 min_mass=1, 
                                #  cutoff_perc= 0, 
                                #  palette = palette, 
                                #  legend_loc="right margin", 
                                 cmap='RdYlBu_r',
                                 size=20, 
                                 alpha=1, 
                                 arrow_size=1,
                                 linewidth=0.15, 
                                 dpi=400, 
                                 save='latent_time2.svg'
                                 )

In [ ]:
df = svdata.var
df = df[(df['fit_likelihood'] > .1) & df['velocity_genes'] == True]

kwargs = dict(xscale='log', fontsize=16)
with scv.GridSpec(ncols=3) as pl:
    pl.hist(df['fit_alpha'], xlabel='transcription rate', **kwargs)
    pl.hist(df['fit_beta'] * df['fit_scaling'], xlabel='splicing rate', xticks=[.1, .4, 1], **kwargs)
    pl.hist(df['fit_gamma'], xlabel='degradation rate', xticks=[.1, .4, 1], **kwargs)

scv.get_df(svdata, 'fit*', dropna=True).head()

In [ ]:
top_genes = svdata.var['fit_likelihood'].sort_values(ascending=False).index[:500]
scv.pl.heatmap(svdata, 
               var_names=top_genes, 
               sortby='latent_time', 
               col_color='celltype', 
               n_convolve=100)

In [ ]:
svdata

In [ ]:
top_genes = svdata.var['fit_likelihood'].sort_values(ascending=False).index
scv.pl.scatter(svdata, basis=top_genes[:15], ncols=5, frameon=False)

In [ ]:
scv.tl.rank_dynamical_genes(svdata, groupby='celltype')
df = scv.get_df(svdata, 'rank_dynamical_genes/names')
df.head(5)

In [ ]:
for celltype in ['NPC', 'IPC', 'OPC', 'ExN_naive', 'InN_naive', 'APC']:
    scv.pl.scatter(svdata, df[celltype][:5], ylabel=celltype, frameon=False)

In [ ]:
svdata.write('ipc_scv.h5ad')

In [ ]:
svdata = sc.read_h5ad('ipc_scv.h5ad')

In [ ]:
svdata

In [ ]:
svdata.obs[['barcode2', 'sample2', 'celltype']].to_csv('IPC_lineage.csv')

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# 定义样本列表
samples = ["GW6", "GW7", "GW9", "GW10", "GW12", "GW16", "GW17", "GW20"]

# 设置PDF保存
plt.rcParams['pdf.fonttype'] = 42  # 确保文字可编辑
plt.rcParams['ps.fonttype'] = 42    # 确保文字可编辑

# 循环绘制每个样本的UMAP图并保存为PDF
for sample in samples:
    # 创建图形
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # 绘制当前样本的UMAP图
    sc.pl.umap(
        svdata, 
        color="sample2", 
        groups=[sample],  # 只显示当前样本
        size=20,          # 点的大小
        frameon=False,    # 去除边框
        title=f"Sample: {sample}",  # 添加标题
        ax=ax,           # 指定绘图的坐标轴
        show=False        # 不立即显示
    )
    
    # 调整布局
    plt.tight_layout()
    
    # 保存为PDF
    plt.savefig(f"./figures/umap_IPC_{sample}.pdf", bbox_inches='tight', dpi=300)
    plt.close()  # 关闭图形，释放内存

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

samples = ["GW6", "GW7", "GW9", "GW10", "GW12", "GW16", "GW17", "GW20"]

my_palette = {
    'NPC': '#faa275', 
    'IPC': '#34a0a4',
    'OPC': '#168aad', 
    'ExN_naive': '#BF3131',
    'InN_naive': '#BA6EAC',
    'APC': '#D15500',
    'Other': '#e0e0e0'  # 浅灰色
}

# 计算统一的 UMAP 坐标范围
umap = svdata.obsm["X_umap"]
xmin, xmax = umap[:, 0].min(), umap[:, 0].max()
ymin, ymax = umap[:, 1].min(), umap[:, 1].max()

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# 子图布局：2 行 4 列
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, sample in zip(axes, samples):

    # ---------- 背景：所有细胞，灰色 ----------
    sc.pl.umap(
        svdata,
        color=None,
        ax=ax,
        show=False,
        frameon=False,
        size=10
    )

    # 手动把背景点设为灰色 & 低层级
    for coll in ax.collections:
        coll.set_color("#e0e0e0")
        coll.set_zorder(1)

    # ---------- 前景：当前时期细胞，按 celltype ----------
    mask = svdata.obs["sample2"] == sample
    sc.pl.umap(
        svdata[mask],
        color="celltype",
        palette=my_palette,
        ax=ax,
        show=False,
        frameon=False,
        size=20
    )

    # 前景点放到最上层
    for coll in ax.collections[-len(np.unique(svdata.obs.loc[mask, "celltype"])):]:
        coll.set_zorder(10)

    # ---------- 坐标 & 外观 ----------
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_title(sample, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig("./figures/umap_celltype_by_stage_single_page.pdf",
            bbox_inches="tight",
            dpi=300)
plt.close()


# scanpy + snapatac2 cells

In [ ]:
anno = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/IPC/atac-meta.csv", index_col=0)
anno

In [ ]:
union = svdata[svdata.obs.index.isin(anno.index)]

In [ ]:
union

In [ ]:
union.obs['UMAP_1'] = union.obsm['X_umap'][:, 0]
union.obs['UMAP_2'] = union.obsm['X_umap'][:, 1]

In [ ]:
union.obs['celltype'] = union.obs['celltype'].astype(str)
# # sample2 为 GW12 且 celltype 为 dIPC -> 改为 dIPC_switching
# mask_switching = (union.obs['celltype'] == 'dIPC') & (union.obs['sample2'] == 'GW12')
# union.obs.loc[mask_switching, 'celltype'] = 'dIPC_switching'

# # sample2 为 GW16, GW17, GW20 且 celltype 为 dIPC -> 改为 dIPC_switched
# target_samples = ['GW16', 'GW17', 'GW20']
# mask_switched = (union.obs['celltype'] == 'dIPC') & (union.obs['sample2'].isin(target_samples))
# union.obs.loc[mask_switched, 'celltype'] = 'dIPC_switched'

union.obs['celltype'] = union.obs['celltype'].astype('category')

In [ ]:
union.obs[['sample2', 'barcode2', 'celltype', 'latent_time', 'UMAP_1', 'UMAP_2']].to_csv('ipc_scv_meta.csv', index=True)

In [ ]:
palette={
    'NPC': '#faa275', 
    'IPC': '#34a0a4',
    'OPC': '#168aad', 
    'ExN_naive': '#BF3131',
    'InN_naive': '#BA6EAC',
    'APC': '#D15500'
}

In [ ]:
sc.pl.umap(
    union,
    color=["celltype"],
    palette=palette,
    legend_loc="on data",
)